# Tripitaka PDF Downloader - Enhanced Version

**This version ensures ALL sections are found and downloaded.**

Improvements:
- More robust HTML parsing with multiple strategies
- Better pagination handling (checks up to 5 pages)
- Detailed logging of what's found
- Handles different HTML structures

Expected sections: Digha Nikaya, Majjhima Nikaya, Samyutta Nikaya, Anguttara Nikaya, Khuddaka Nikaya, Vinaya, Abhidhamma

In [1]:
!pip install -q requests beautifulsoup4 gdown tqdm lxml html5lib

In [2]:
import requests
from bs4 import BeautifulSoup
import gdown
import os
import re
import time
from urllib.parse import urlparse
from tqdm.auto import tqdm
import json
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

In [3]:
# Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print('✓ Drive mounted')
except:
    print('Not on Colab or already mounted')

Mounted at /content/drive
✓ Drive mounted


In [4]:
# Configuration
BASE_URL = 'https://download.ifbcnet.org/category/thripitaka-books/'
PROJECT_DIR = '/content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/'
TRIPITAKA_DIR = os.path.join(PROJECT_DIR, 'data/raw/tripitaka')

os.makedirs(TRIPITAKA_DIR, exist_ok=True)

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36'
}

print(f'📁 Output: {TRIPITAKA_DIR}')

📁 Output: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka


## Helper Functions

In [5]:
def clean_section_name(url):
    path = urlparse(url).path
    parts = [p for p in path.split('/') if p and not re.match(r'^\d{2,4}$', p)]
    if parts:
        name = parts[-1]
        name = re.sub(r'%[0-9a-fA-F]{2}', '', name)
        name = re.sub(r'[^a-zA-Z0-9-_]', '', name)
        return name if name else 'unknown'
    return 'unknown'

def extract_google_drive_id(url):
    patterns = [
        r'drive\\.google\\.com/file/d/([a-zA-Z0-9_-]+)',
        r'drive\\.google\\.com/open\\?id=([a-zA-Z0-9_-]+)',
        r'/d/([a-zA-Z0-9_-]+)',
    ]
    for p in patterns:
        m = re.search(p, url)
        if m:
            return m.group(1)
    return None

def sanitize_filename(name):
    return re.sub(r'[<>:"/\\\|?*]', '_', name).strip('. ')

def download_from_google_drive(file_id, output_path, max_retries=3):
    for attempt in range(max_retries):
        try:
            url = f'https://drive.google.com/uc?id={file_id}'
            gdown.download(url, output_path, quiet=False)
            if os.path.exists(output_path) and os.path.getsize(output_path) > 0:
                return True
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    return False

## Enhanced Section Discovery

In [6]:
def get_all_sections(base_url, max_pages=5):
    """Scan all pages to find every Tripitaka section."""
    print('='*80)
    print('SCANNING FOR ALL TRIPITAKA SECTIONS')
    print('='*80)

    all_sections = []
    seen = set()

    for page_num in range(1, max_pages + 1):
        url = base_url if page_num == 1 else f'{base_url}page/{page_num}/'
        print(f'\nPage {page_num}: {url}')

        try:
            resp = requests.get(url, headers=HEADERS, timeout=30)
            if resp.status_code == 404:
                print(f'  ⚠️  Page {page_num} not found, stopping')
                break
            resp.raise_for_status()

            soup = BeautifulSoup(resp.content, 'html.parser')

            # Try multiple container strategies
            containers = soup.find_all('article')
            if not containers:
                containers = soup.find_all('div', class_=re.compile(r'post|entry|item'))
            if not containers:
                containers = soup.find_all('div', id=re.compile(r'post-\d+'))

            found_on_page = 0

            for container in containers:
                # Look for title and link
                title_tag = container.find(['h1', 'h2', 'h3'], class_=re.compile(r'title|entry'))
                if not title_tag:
                    title_tag = container.find(['h1', 'h2', 'h3'])

                link_tag = None
                if title_tag:
                    link_tag = title_tag.find('a', href=True)
                if not link_tag:
                    link_tag = container.find('a', href=True)

                if link_tag:
                    href = link_tag['href']

                    # Check if it's a Tripitaka section
                    if any(kw in href.lower() for kw in ['thripitaka', 'tripitaka', 'tipitaka']):
                        if href not in seen:
                            title = link_tag.get_text(strip=True)
                            if not title and title_tag:
                                title = title_tag.get_text(strip=True)

                            section_name = clean_section_name(href)

                            all_sections.append({
                                'url': href,
                                'title': title or section_name,
                                'section_name': section_name
                            })
                            seen.add(href)
                            found_on_page += 1

            print(f'  ✓ Found {found_on_page} sections')

            if found_on_page == 0:
                print(f'  No sections on page {page_num}, stopping')
                break

            time.sleep(1.5)

        except Exception as e:
            print(f'  ❌ Error: {e}')
            break

    print(f'\n{'='*80}')
    print(f'TOTAL SECTIONS FOUND: {len(all_sections)}')
    print('='*80)

    if all_sections:
        print('\nAll sections:')
        for i, s in enumerate(all_sections, 1):
            print(f'{i:2d}. {s["section_name"]}')
            print(f'    Title: {s["title"][:65]}')

    return all_sections

In [7]:
def get_pdfs_from_section(section_url):
    """Extract PDF links from section page."""
    try:
        resp = requests.get(section_url, headers=HEADERS, timeout=30)
        resp.raise_for_status()
        soup = BeautifulSoup(resp.content, 'html.parser')

        pdfs = []
        for link in soup.find_all('a', href=True):
            href = link['href']
            if 'drive.google.com' in href:
                file_id = extract_google_drive_id(href)
                if file_id:
                    title = link.get_text(strip=True) or f'file_{file_id}'
                    pdfs.append({'file_id': file_id, 'url': href, 'title': title})

        time.sleep(0.5)
        return pdfs
    except Exception as e:
        print(f'Error: {e}')
        return []

## Download Functions

In [8]:
def download_section(section_info, pdf_links):
    """Download all PDFs for one section."""
    section_name = section_info['section_name']
    section_dir = os.path.join(TRIPITAKA_DIR, section_name)
    os.makedirs(section_dir, exist_ok=True)

    stats = {'section': section_name, 'total': len(pdf_links),
             'successful': 0, 'failed': 0, 'skipped': 0, 'files': []}

    for pdf in tqdm(pdf_links, desc=f'{section_name}'):
        title = sanitize_filename(pdf['title'])
        filename = f'{title}.pdf' if not title.endswith('.pdf') else title
        filepath = os.path.join(section_dir, filename)

        if os.path.exists(filepath):
            stats['skipped'] += 1
            stats['files'].append({'name': filename, 'status': 'skipped'})
            continue

        if download_from_google_drive(pdf['file_id'], filepath):
            stats['successful'] += 1
            stats['files'].append({'name': filename, 'status': 'success',
                                   'size': os.path.getsize(filepath)})
        else:
            stats['failed'] += 1
            stats['files'].append({'name': filename, 'status': 'failed'})

        time.sleep(1)

    return stats

## Main Pipeline

In [9]:
def main():
    """Complete download pipeline."""
    start = datetime.now()
    print(f'\n🚀 Starting at {start.strftime("%H:%M:%S")}\n')

    # Find all sections
    sections = get_all_sections(BASE_URL, max_pages=5)

    if not sections:
        print('\n❌ No sections found!')
        return

    # Save section list
    with open(os.path.join(TRIPITAKA_DIR, '_sections.json'), 'w') as f:
        json.dump(sections, f, indent=2, ensure_ascii=False)

    # Download each section
    results = []

    for i, section in enumerate(sections, 1):
        print(f'\n{'='*80}')
        print(f'Section {i}/{len(sections)}: {section["title"]}')
        print(f'URL: {section["url"]}')
        print('='*80)

        pdfs = get_pdfs_from_section(section['url'])
        print(f'Found {len(pdfs)} PDFs')

        if pdfs:
            result = download_section(section, pdfs)
            results.append(result)
        else:
            print('⚠️  No PDFs in this section')

        # Save progress
        with open(os.path.join(TRIPITAKA_DIR, '_progress.json'), 'w') as f:
            json.dump(results, f, indent=2)

    # Summary
    print(f'\n{'='*80}')
    print('FINAL SUMMARY')
    print('='*80)

    total = sum(r['total'] for r in results)
    success = sum(r['successful'] for r in results)
    failed = sum(r['failed'] for r in results)
    skipped = sum(r['skipped'] for r in results)

    print(f'Sections: {len(results)}')
    print(f'Total PDFs: {total}')
    print(f'✓ Downloaded: {success}')
    print(f'⊝ Skipped: {skipped}')
    print(f'✗ Failed: {failed}')

    end = datetime.now()
    print(f'\n⏱️  Duration: {end - start}')

    # Final report
    report = {
        'timestamp': end.isoformat(),
        'duration': str(end - start),
        'summary': {'sections': len(results), 'total': total,
                    'success': success, 'failed': failed, 'skipped': skipped},
        'details': results
    }

    with open(os.path.join(TRIPITAKA_DIR, '_report.json'), 'w') as f:
        json.dump(report, f, indent=2)

    print(f'\n✅ Complete! Report: {TRIPITAKA_DIR}/_report.json')

## Execute

In [10]:
# RUN THE COMPLETE DOWNLOADER
main()


🚀 Starting at 09:18:32

SCANNING FOR ALL TRIPITAKA SECTIONS

Page 1: https://download.ifbcnet.org/category/thripitaka-books/
  ✓ Found 1 sections

Page 2: https://download.ifbcnet.org/category/thripitaka-books/page/2/
  ✓ Found 0 sections
  No sections on page 2, stopping

TOTAL SECTIONS FOUND: 1

All sections:
 1. --thripitaka-digha-nikaya
    Title: ත්‍රිපිටකය – දීඝනිකාය ( thripitaka – Digha Nikaya)

Section 1/1: ත්‍රිපිටකය – දීඝනිකාය ( thripitaka – Digha Nikaya)
URL: https://download.ifbcnet.org/2015/02/10/%e0%b6%ad%e0%b7%8a%e2%80%8d%e0%b6%bb%e0%b7%92%e0%b6%b4%e0%b7%92%e0%b6%a7%e0%b6%9a%e0%b6%ba-%e0%b6%af%e0%b7%93%e0%b6%9d%e0%b6%b1%e0%b7%92%e0%b6%9a%e0%b7%8f%e0%b6%ba-thripitaka-digha-nikaya/
Found 1 PDFs


--thripitaka-digha-nikaya:   0%|          | 0/1 [00:00<?, ?it/s]

Downloading...
From (original): https://drive.google.com/uc?id=0B5RD8Fve5lhmeTNhUU8zQ0Z1LUE
From (redirected): https://drive.google.com/uc?id=0B5RD8Fve5lhmeTNhUU8zQ0Z1LUE&confirm=t&uuid=2576cbc8-ca96-4b51-bdb9-92dee91bfb10
To: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/--thripitaka-digha-nikaya/දීඝ නිකාය 1.pdf

  0%|          | 0.00/140M [00:00<?, ?B/s]
  3%|▎         | 4.72M/140M [00:00<00:03, 37.0MB/s]
  6%|▋         | 8.91M/140M [00:00<00:05, 25.8MB/s]
 12%|█▏        | 17.3M/140M [00:00<00:03, 34.5MB/s]
 18%|█▊        | 24.6M/140M [00:00<00:02, 44.8MB/s]
 21%|██▏       | 29.9M/140M [00:00<00:02, 42.4MB/s]
 25%|██▌       | 35.7M/140M [00:00<00:02, 45.3MB/s]
 30%|██▉       | 41.9M/140M [00:00<00:01, 49.6MB/s]
 35%|███▌      | 49.3M/140M [00:01<00:01, 48.2MB/s]
 42%|████▏     | 58.7M/140M [00:01<00:01, 59.7MB/s]
 47%|████▋     | 65.5M/140M [00:01<00:01, 44.1MB/s]
 57%|█████▋    | 79.7M/140M [00:01<00:01, 51.6MB/s]
 62%|██████▏   


FINAL SUMMARY
Sections: 1
Total PDFs: 1
✓ Downloaded: 1
⊝ Skipped: 0
✗ Failed: 0

⏱️  Duration: 0:00:09.602290

✅ Complete! Report: /content/drive/Othercomputers/My Mac/Multi-lingual-Buddhist-Conversational-Agent/data/raw/tripitaka/_report.json


## Post-Download Analysis

In [ ]:
# Analyze what was downloaded
import glob

sections = [d for d in os.listdir(TRIPITAKA_DIR)
            if os.path.isdir(os.path.join(TRIPITAKA_DIR, d)) and not d.startswith('_')]

print('\nDownloaded Sections:')
print('='*60)

for sec in sorted(sections):
    path = os.path.join(TRIPITAKA_DIR, sec)
    pdfs = glob.glob(os.path.join(path, '*.pdf'))
    size = sum(os.path.getsize(p) for p in pdfs) / (1024*1024)
    print(f'{sec}:')
    print(f'  Files: {len(pdfs)}, Size: {size:.1f} MB')